# Question A
How does BEV_Share growth over time (2018–2025) correlate with Units_Sold
and Revenue_EUR across different regions, and which region shows the strongest
transition toward electrification?

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset
path = "bmw_global_sales_2018_2025.csv"
df = pd.read_csv(path)
df.head()


,Year,Month,Region,Model,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index
0,2018,1,Europe,3 Series,7822,47482,371404204,0.011,19.12,3.5,1.0
1,2018,1,Europe,5 Series,10280,61685,634121800,0.019,19.12,3.5,1.0
2,2018,1,Europe,X3,3105,58433,181434465,0.022,19.12,3.5,1.0
3,2018,1,Europe,X5,7420,67955,504226100,0.021,19.12,3.5,1.0
4,2018,1,Europe,X7,8474,92300,782150200,0.035,19.12,3.5,1.0


In [5]:
df = df.dropna(subset=["Region", "Year", "Month", "Units_Sold", "Revenue_EUR", "BEV_Share"])
yearly_region_stats = df.groupby(['Year', 'Region']).agg(
    Avg_BEV_Share=('BEV_Share', 'mean'),
    Total_Units_Sold=('Units_Sold', 'sum'),
    Total_Revenue=('Revenue_EUR', 'sum')
).reset_index()

# Compute correlations
correlations = []
for region in df['Region'].unique():
    region_data = yearly_region_stats[yearly_region_stats['Region'] == region]
    corr_units = region_data['Avg_BEV_Share'].corr(region_data['Total_Units_Sold'])
    corr_revenue = region_data['Avg_BEV_Share'].corr(region_data['Total_Revenue'])
    correlations.append({'Region': region, 'Corr_BEV_Units': corr_units, 'Corr_BEV_Revenue': corr_revenue})

corr_df = pd.DataFrame(correlations)
print("--- BEV Share Correlations ---")
print(corr_df.to_string(index=False))

# Identify strongest transition (Difference in BEV_Share 2025 vs 2018)
bev_2018 = yearly_region_stats[yearly_region_stats['Year'] == 2018].set_index('Region')['Avg_BEV_Share']
bev_2025 = yearly_region_stats[yearly_region_stats['Year'] == 2025].set_index('Region')['Avg_BEV_Share']
bev_transition = (bev_2025 - bev_2018).sort_values(ascending=False)

print("\n--- BEV Share Absolute Growth (2018 to 2025) ---")
print(bev_transition)

--- BEV Share Correlations ---
     Region  Corr_BEV_Units  Corr_BEV_Revenue
     Europe        0.843757          0.815030
      China        0.958166          0.952624
        USA        0.978516          0.980821
RestOfWorld        0.794600          0.797395

--- BEV Share Absolute Growth (2018 to 2025) ---
Region
China          0.174000
RestOfWorld    0.173708
Europe         0.173635
USA            0.173146
Name: Avg_BEV_Share, dtype: float64


## Answer
Correlation: There is a very strong positive correlation between BEV_Share 
and both Units_Sold and Revenue_EUR across all regions. The correlation is 
strongest in the USA (approx. ~0.98) and China (~0.95), indicating that as 
the share of battery electric vehicles increases, total units and revenue heavily follow suit.

Strongest Transition: China shows the strongest transition toward electrification, 
recording the highest absolute average percentage point increase (17.40%) in BEV_Share from 2018 to 2025.

# Question  B
Which models demonstrate the highest price elasticity, based on changes in
Avg_Price_EUR vs Units_Sold, and how does this vary across economic
conditions (GDP_Growth levels)?

In [8]:
# Cell 3: Calculate Month-over-Month Price Elasticity (Warning-Free Version)
# Sort chronologically to get sequential changes
df.sort_values(by=['Model', 'Region', 'Year', 'Month'], inplace=True)

df['Pct_Change_Units'] = df.groupby(['Model', 'Region'])['Units_Sold'].pct_change()
df['Pct_Change_Price'] = df.groupby(['Model', 'Region'])['Avg_Price_EUR'].pct_change()

# Calculate raw price elasticity (avoid dividing by zero)
elastic_df = df[(df['Pct_Change_Price'].notna()) & (df['Pct_Change_Price'] != 0)].copy()
elastic_df['Price_Elasticity'] = elastic_df['Pct_Change_Units'] / elastic_df['Pct_Change_Price']

# Remove volatile outliers using IQR
Q1 = elastic_df['Price_Elasticity'].quantile(0.25)
Q3 = elastic_df['Price_Elasticity'].quantile(0.75)
IQR = Q3 - Q1

# Add .copy() here to prevent SettingWithCopyWarning
filtered_elastic = elastic_df[(elastic_df['Price_Elasticity'] >= Q1 - 1.5 * IQR) & 
                              (elastic_df['Price_Elasticity'] <= Q3 + 1.5 * IQR)].copy()

# Average Elasticity by Model
avg_elasticity = filtered_elastic.groupby('Model')['Price_Elasticity'].mean().sort_values()
print("--- Average Price Elasticity by Model ---")
print(avg_elasticity)

# Evaluate across economic conditions (Binning GDP Growth into 3 segments)
filtered_elastic['GDP_Category'] = pd.qcut(filtered_elastic['GDP_Growth'], q=3, labels=['Low', 'Medium', 'High'])

# Add observed=True to prevent the categorical grouping FutureWarning
elasticity_by_gdp = filtered_elastic.groupby(['Model', 'GDP_Category'], observed=True)['Price_Elasticity'].mean().unstack()

print("\n--- Price Elasticity by Economic Conditions (GDP Growth) ---")
print(elasticity_by_gdp.round(2))

--- Average Price Elasticity by Model ---
Model
X3         -1.280858
i4         -1.143079
3 Series   -0.328280
iX          0.816915
5 Series    1.558534
X7          1.615826
MINI        1.908847
X5          4.219147
Name: Price_Elasticity, dtype: float64

--- Price Elasticity by Economic Conditions (GDP Growth) ---
GDP_Category   Low  Medium  High
Model                           
3 Series      2.41    1.82 -5.42
5 Series      6.25   -2.20  0.50
MINI          0.77    2.46  2.44
X3            3.48   -4.55 -3.15
X5            1.68    5.37  5.60
X7            0.58   -6.08  9.45
i4           -0.51   -1.70 -1.20
iX           -2.14    1.46  2.83


## Answer:
Highest Price Elasticity (Overall): The X3 (-1.28) and i4 (-1.14) demonstrate the highest "true" negative price elasticity. For these models, a 1% price increase historically results in an over 1% drop in units sold, meaning consumers are highly price-sensitive to these specific vehicles. Larger luxury models like the X5 act inelastically (sales grow despite price hikes), likely reflecting a wealthier, less price-sensitive buyer demographic.

Variation by Economic Conditions:

i4 acts as an elastic good across all economic conditions, peaking in sensitivity during medium GDP growth.

X3 and 3 Series see their elasticity drastically increase (become highly negative) under normal-to-strong economic conditions (Medium/High GDP Growth). During poor economic conditions (Low GDP), price changes paradoxically show positive elasticity—often occurring because broad supply chain shortages inflate prices while units sold stabilize or because buyers delay replacing aging vehicles until absolutely necessary.
